# Diagnosis for the LS-SSDD YOLO model

**Inputs to attach:**
1. The preprocessing notebook's Output (`lsssdd_yolo/`)
2. The training notebook's Output (the one containing `weights/best.pt`)
3. *(optional)* the raw LS-SSDD dataset, for the image-size check

A GPU helps but isn't required. This notebook only runs predictions.

**What it checks**
1. **Raw data sanity:** does each tile's real size match the size written in its XML? If the Kaggle copy was resized, every box would be off.
2. **Stretch values:** the brightness settings used in preprocessing.
3. **Recall by ship size:** are the misses only the tiny ships, or big ones too?
4. **Recall at a loose vs. strict overlap:** if the model finds ships but its boxes are shifted, recall jumps a lot when the overlap requirement is loosened. That would point to a label-alignment bug, not a weak model.
5. **Missed-ship gallery:** tiles with true ships (green), predictions (red) and **misses (yellow)**.

In [ ]:
!pip install -q ultralytics

In [ ]:
import re, gc, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
import numpy as np, cv2, matplotlib.pyplot as plt, torch
from ultralytics import YOLO, settings
settings.update({"raytune": False, "mlflow": False, "comet": False, "clearml": False, "dvc": False, "wandb": False, "sync": False})

INPUT_ROOT = Path("/kaggle/input")
OUT        = Path("/kaggle/working/diagnosis"); OUT.mkdir(parents=True, exist_ok=True)
SPLIT      = "test"      # "test" (scenes 11-15) or "val" (scene 10)
IMGSZ      = 800
CONF       = 0.25        # threshold for counting a prediction as "found"
DEVICE     = 0 if torch.cuda.is_available() else "cpu"

data_dir = next(p.parent for p in INPUT_ROOT.rglob("data.yaml") if (p.parent / "images" / "train").is_dir())
print("dataset:", data_dir)
weights = sorted(set(INPUT_ROOT.rglob("best.pt"))) or sorted(set(INPUT_ROOT.rglob("*.pt")))
if not weights:
    print("Attached inputs (2 levels deep):")
    for p in sorted(INPUT_ROOT.glob("*/*")) + sorted(INPUT_ROOT.glob("*/*/*")):
        print("  ", p)
    raise SystemExit("No .pt weights found under /kaggle/input. Attach the TRAINING notebook's Output "
                     "(the run that contains runs/<name>/weights/best.pt), or set WEIGHTS to the path by hand.")
for i, w in enumerate(weights): print(f"weights[{i}]:", w)
WEIGHTS = weights[-1]     # change the index if several runs are attached, e.g. pick the P2 run
print("using:", WEIGHTS)

## 1. Raw data sanity: real image size vs. XML size
This only runs if the raw LS-SSDD dataset is attached. Every tile should be 800×800, both in the image and in its XML.

In [ ]:
TILE_RE = re.compile(r"^\d+_\d+_\d+$")
raw_imgs = {p.stem: p for p in INPUT_ROOT.rglob("*.jpg") if TILE_RE.match(p.stem)}
raw_xmls = {p.stem: p for p in INPUT_ROOT.rglob("*.xml") if TILE_RE.match(p.stem)}
common = sorted(set(raw_imgs) & set(raw_xmls))
if not common:
    print("raw dataset not attached, skipping")
else:
    rng = np.random.default_rng(0); mism = 0; sizes = Counter(); xml_sizes = Counter()
    sample = rng.choice(common, min(300, len(common)), replace=False)
    for s in sample:
        h, w = cv2.imread(str(raw_imgs[s]), cv2.IMREAD_GRAYSCALE).shape
        size = ET.parse(raw_xmls[s]).getroot().find("size")
        xw, xh = int(size.find("width").text), int(size.find("height").text)
        sizes[(w, h)] += 1; xml_sizes[(xw, xh)] += 1; mism += (w, h) != (xw, xh)
    print("real image sizes:", dict(sizes)); print("XML sizes:       ", dict(xml_sizes))
    print(f"MISMATCHES: {mism} of {len(sample)} checked  <-- must be 0")

## 2. Stretch values used in preprocessing

In [ ]:
st = data_dir / "stretch.txt"
print(st.read_text() if st.exists() else "stretch.txt not found")
im = cv2.imread(str(next((data_dir / "images" / SPLIT).glob("*"))), cv2.IMREAD_GRAYSCALE)
print("sample preprocessed tile:", im.shape, "min/median/max:", im.min(), np.median(im), im.max(),
      " | share of pixels at 255 (saturated): %.3f%%" % ((im == 255).mean() * 100))

## 3–4. Recall by ship size, at loose and strict overlap
Every true ship is matched to the best-overlapping prediction (confidence ≥ `CONF`). A ship counts as **found at IoU t** if that overlap is at least t.
- **IoU 0.5** is what mAP50 uses.
- **IoU 0.1** means "the model put a box roughly on the ship".

If recall at 0.1 is much higher than at 0.5, the model sees the ships but its boxes are misplaced or the wrong size. That points to a label or scaling problem. If both are low for small ships only, the model simply can't see the tiny ships.

In [ ]:
def load_gt(lbl, w, h):
    out = []
    for ln in lbl.read_text().split("\n") if lbl.exists() else []:
        if ln.strip():
            _, cx, cy, bw, bh = map(float, ln.split())
            out.append([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h])
    return np.array(out).reshape(-1, 4)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

model = YOLO(str(WEIGHTS))
img_paths = sorted((data_dir / "images" / SPLIT).glob("*"))
records, n_pred, n_fp, per_tile = [], 0, 0, {}
# Predict in small chunks and free memory after each one, so RAM stays flat on Kaggle.
CHUNK = 16
for i0 in range(0, len(img_paths), CHUNK):
    chunk = [str(p) for p in img_paths[i0:i0 + CHUNK]]
    with torch.no_grad():
        results = model.predict(chunk, imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)
    for r in results:
        p = Path(r.path); h, w = r.orig_shape
        gt = load_gt(data_dir / "labels" / SPLIT / (p.stem + ".txt"), w, h)
        pr = r.boxes.xyxy.cpu().numpy()
        M = iou_matrix(gt, pr)
        best = M.max(1) if M.size else np.zeros(len(gt))
        n_pred += len(pr); n_fp += int((M.max(0) < 0.5).sum()) if M.size else len(pr)
        for g, b in zip(gt, best):
            records.append((np.sqrt((g[2]-g[0]) * (g[3]-g[1])), b))
        per_tile[p] = (gt, pr, best)
    del results; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if (i0 // CHUNK) % 25 == 0:
        print(f"  {min(i0 + CHUNK, len(img_paths))}/{len(img_paths)} tiles done")

rec = np.array(records)
side, best_iou = rec[:, 0], rec[:, 1]
bins = [(0, 8), (8, 12), (12, 16), (16, 24), (24, 32), (32, 1e9)]
print(f"{SPLIT}: {len(img_paths)} tiles, {len(rec)} ships, {n_pred} predictions at conf>={CONF}, "
      f"{n_fp} of them match no ship (false alarms)\n")
print(f"{'ship size (px)':>15} {'ships':>6} {'share':>6} {'recall@0.5':>11} {'recall@0.3':>11} {'recall@0.1':>11}")
rows = []
for lo, hi in bins:
    m = (side >= lo) & (side < hi)
    if m.sum() == 0: continue
    r5, r3, r1 = [(best_iou[m] >= t).mean() for t in (0.5, 0.3, 0.1)]
    rows.append((f"{lo}-{hi if hi < 1e9 else '+'}", m.sum(), r5, r3, r1))
    print(f"{rows[-1][0]:>15} {m.sum():6d} {m.mean()*100:5.0f}% {r5:11.2f} {r3:11.2f} {r1:11.2f}")
print(f"{'ALL':>15} {len(rec):6d} {'100%':>6} {(best_iou>=.5).mean():11.2f} {(best_iou>=.3).mean():11.2f} {(best_iou>=.1).mean():11.2f}")

fig, ax = plt.subplots(figsize=(8, 4)); x = np.arange(len(rows))
for k, (t, c) in enumerate([("IoU 0.5", 2), ("IoU 0.3", 3), ("IoU 0.1", 4)]):
    ax.bar(x + (k - 1) * 0.27, [r[c] for r in rows], 0.27, label=t)
ax.set_xticks(x, [r[0] for r in rows]); ax.set_xlabel("ship size, sqrt(w*h) in px"); ax.set_ylabel("recall")
ax.set_ylim(0, 1); ax.legend(); ax.set_title(f"Recall by ship size ({SPLIT}, conf>={CONF})")
plt.tight_layout(); fig.savefig(OUT / "recall_by_size.png", dpi=110); plt.show()

## 5. Missed-ship gallery
The 4 tiles with the most missed ships. **Green** = true ship found, **yellow** = true ship missed, **red** = prediction.

In [ ]:
worst = sorted(per_tile.items(), key=lambda kv: -int((kv[1][2] < 0.5).sum()))[:4]
fig, axes = plt.subplots(1, 4, figsize=(24, 6.5))
for ax, (p, (gt, pr, best)) in zip(axes, worst):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    for x1, y1, x2, y2 in pr.astype(int):
        cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 2)
    for (x1, y1, x2, y2), b in zip(gt.astype(int), best):
        cv2.rectangle(im, (x1-3, y1-3), (x2+3, y2+3), (0, 255, 0) if b >= 0.5 else (255, 255, 0), 1)
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{p.stem}: {len(gt)} ships, {int((best < .5).sum())} missed, {len(pr)} predicted", fontsize=10)
plt.tight_layout(); fig.savefig(OUT / "missed_gallery.png", dpi=110); plt.show()